# Instalación depedencias

In [ ]:
!pip -q uninstall -y rapids-dask-dependency
!pip install segmentation-models-pytorch==0.3.3 albumentations==1.4.6 torchmetrics==1.4.0 tqdm==4.66.4 rasterio==1.3.10
!pip -q uninstall -y dask distributed
!pip -q install pystac-client==0.7.6 stackstac==0.5.0 odc-stac==0.3.9 \
               geopandas==1.0.1 shapely==2.0.4 rasterio==1.3.10 rioxarray==0.15.5 \
               rasterstats==0.19.0 planetary-computer==1.0.0 \
               segmentation-models-pytorch==0.3.3 albumentations==1.4.6 torchmetrics==1.4.0 \
               "dask[complete]==2024.10.0" "distributed==2024.10.0"

# Librerias

In [ ]:
from pathlib import Path
import geopandas as gpd
import shapely.geometry as sgeom
from pystac_client import Client
import stackstac, numpy as np, xarray as xr
import rioxarray as rxr
from pathlib import Path
import xarray as xr
import numpy as np
from rasterstats import zonal_stats
import pandas as pd
import rasterio
from rasterio.windows import Window

import torch, numpy as np, matplotlib.pyplot as plt, rasterio
from torch.utils.data import Dataset, DataLoader
from segmentation_models_pytorch import Unet
from torchmetrics import JaccardIndex
from tqdm import tqdm


# Ejemplo para cargar polígonos en formato (GeoJSON/SHP)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = Path('/content/drive/MyDrive/geotermia_colombia_noGEE'); PROJ.mkdir(parents=True, exist_ok=True)

# Cambiar dirección
POLY_PATH = PROJ/'zonas_geotermicas.geojson'  # o .shp
gdf = gpd.read_file(POLY_PATH).to_crs(4326)

AOI = sgeom.mapping(gdf.unary_union)
T1_START, T1_END = '2018-01-01', '2018-12-31'
T2_START, T2_END = '2024-01-01', '2024-12-31'


# Ejemplo Buscar imágenes satelitales en  Sentinel-2 L2A

In [ ]:

STAC_URL = "https://earth-search.aws.element84.com/v1"
catalog = Client.open(STAC_URL)

def search_s2(aoi, start, end, max_cc=40):
    s = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects=aoi,
        datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lt": max_cc}},
        limit=200
    )
    return list(s.get_items())

items_t1 = search_s2(AOI, T1_START, T1_END)
items_t2 = search_s2(AOI, T2_START, T2_END)
print("T1:", len(items_t1), "T2:", len(items_t2))
assert len(items_t1)>0 and len(items_t2)>0, "No se encontraron escenas (ajusta fechas o AOI)."


## Preprocesamiento

Donde las imágenes satelitales crudas se corrigen por distorsiones atmósfericas, como la cobertura nubosa y diferencias radiométricas

*   Normalización de reflectancia para cada píxel
*   Enmascaramiento de nubes



In [ ]:

BANDS = ["red","nir","green","scl"]  # bandas S2 L2A

def stack_period(items):
    da = stackstac.stack(items, assets=BANDS, resolution=10, dtype="float32", fill_value=np.nan)
    # Escalar reflectancias 0..1
    for b in ["red","nir","green"]:
        da.loc[dict(band=b)] = da.loc[dict(band=b)]/10000.0
    return da

def mask_scl(da):
    scl = da.loc[dict(band="scl")]
    good = xr.where((scl==4)|(scl==5)|(scl==6)|(scl==11), 1, np.nan)
    out = da.copy()
    for b in ["red","nir","green"]:
        out.loc[dict(band=b)] = out.loc[dict(band=b)] * good
    return out.drop_sel(band="scl")

da_t1 = mask_scl(stack_period(items_t1))
da_t2 = mask_scl(stack_period(items_t2))

m_t1 = da_t1.median(dim="time", skipna=True)   # composición temporal
m_t2 = da_t2.median(dim="time", skipna=True)
m_t1, m_t2


Bandas disponibles por el satelite Sentinel

*   Banda red
*   NIR
*   Green
*   scl

Se utiliza la banda espectral que captura información específica del infrarrojo cercano, sensible a la vegetación.


In [ ]:
BANDS = ["red","nir","green","scl"]  # nombres estándar en S2 L2A

def stack_period(items):
    da = stackstac.stack(
        items, assets=BANDS,
        resolution=10, dtype="float32",
        fill_value=np.nan
    )
    # Escalar reflectancias a 0..1
    for b in ["red","nir","green"]:
        da.loc[dict(band=b)] = da.loc[dict(band=b)] / 10000.0
    return da

da_t1 = stack_period(items_t1)
da_t2 = stack_period(items_t2)

# Enmascarar nubes/sombras
def mask_scl(da):
    scl = da.loc[dict(band="scl")]
    good = xr.where((scl==4)|(scl==5)|(scl==6)|(scl==11), 1, np.nan)
    out = da.copy()
    for b in ["red","nir","green"]:
        out.loc[dict(band=b)] = out.loc[dict(band=b)] * good
    return out.drop_sel(band="scl")

m_t1 = mask_scl(da_t1).median(dim="time", skipna=True)
m_t2 = mask_scl(da_t2).median(dim="time", skipna=True)

def ndvi(med):
    nir = med.loc[dict(band="nir")]
    red = med.loc[dict(band="red")]
    return (nir - red) / (nir + red + 1e-6)

ndvi_t1 = ndvi(m_t1).rename("NDVI_t1")
ndvi_t2 = ndvi(m_t2).rename("NDVI_t2")
dndvi   = (ndvi_t2 - ndvi_t1).rename("dNDVI")

float(ndvi_t1.median().compute().values), float(dndvi.median().compute().values)


# **Exportar GeoTIFFs a Drive**

Formato de imágen basado en Tiff que incorpora metadatos georeferenciados


In [ ]:
NDVI_T1_TIF = PROJ/"NDVI_t1.tif"
NDVI_T2_TIF = PROJ/"NDVI_t2.tif"
DNDVI_TIF   = PROJ/"dNDVI.tif"

def to_geotiff(da, path):

    out = da.expand_dims(band=[da.name or "band1"]) if 'band' not in da.dims else da
    out = out.rio.write_crs("EPSG:4326")
    out.rio.to_raster(str(path))
    print("Guardado:", path)

to_geotiff(ndvi_t1, NDVI_T1_TIF)
to_geotiff(ndvi_t2, NDVI_T2_TIF)
to_geotiff(dndvi,   DNDVI_TIF)


# **Estadísticas zonales por polígono (NDVI)**

In [ ]:
zs = zonal_stats(
    vectors=gdf, raster=str(DNDVI_TIF),
    stats=["mean","median","std","percentile_10","percentile_90"],
    nodata=np.nan, geojson_out=False
)
df = pd.DataFrame(zs)
out = pd.concat([gdf.drop(columns="geometry").reset_index(drop=True), df], axis=1)
out_path = PROJ/"zonal_stats_dNDVI.xlsx"
out.to_excel(out_path, index=False)
out.head()


# **Generar máscara de cambio (0 ↓, 1 =, 2 ↑)**

In [ ]:
tau = 0.05  # umbral de cambio
dndvi = rxr.open_rasterio(PROJ/"dNDVI.tif").squeeze()

cls = xr.where(dndvi < -tau, 0, xr.where(dndvi > tau, 2, 1))
cls.name = "change_class"
CLS_TIF = PROJ/"mask_change.tif"
cls = cls.rio.write_crs("EPSG:4326")
cls.rio.to_raster(str(CLS_TIF))
print("Máscara guardada en:", CLS_TIF)


# **Crear un stack simple de 3 canales (NDVIt1, NDVIt2, ANDVI)**

In [ ]:
ndvi_t1 = rxr.open_rasterio(PROJ/"NDVI_t1.tif").squeeze()
ndvi_t2 = rxr.open_rasterio(PROJ/"NDVI_t2.tif").squeeze()
stack3 = xr.concat([ndvi_t1, ndvi_t2, dndvi], dim="band")
stack3 = stack3.assign_coords(band=["NDVI_t1","NDVI_t2","dNDVI"])
STACK3_TIF = PROJ/"stack3.tif"
stack3.rio.write_crs("EPSG:4326").rio.to_raster(str(STACK3_TIF))
print("Stack guardado en:", STACK3_TIF)


# **Cortar en tiles 256×256**

In [ ]:
IMG_DIR = PROJ/"tiles_stack3"
MSK_DIR = PROJ/"tiles_mask"
IMG_DIR.mkdir(exist_ok=True); MSK_DIR.mkdir(exist_ok=True)
size = 256

with rasterio.open(STACK3_TIF) as srcI, rasterio.open(CLS_TIF) as srcM:
    H, W = srcI.height, srcI.width
    k = 0
    for y in range(0, H, size):
        for x in range(0, W, size):
            if y + size > H or x + size > W:
                continue
            w = Window(x, y, size, size)
            img = srcI.read(window=w)  # (3,256,256)
            msk = srcM.read(1, window=w)
            if np.isnan(img).all():
                continue
            profileI = srcI.profile.copy()
            profileI.update(height=size, width=size, transform=srcI.window_transform(w), count=3)
            profileM = srcM.profile.copy()
            profileM.update(height=size, width=size, transform=srcM.window_transform(w), count=1)
            with rasterio.open(IMG_DIR/f"tile_{k:05d}.tif", "w", **{**profileI, "dtype":"float32"}) as dst:
                dst.write(img.astype("float32"))
            with rasterio.open(MSK_DIR/f"mask_{k:05d}.tif", "w", **{**profileM, "dtype":"uint8"}) as dst:
                dst.write(msk.astype("uint8"), 1)
            k += 1
print("Tiles creados:", k)


# **Entrenamiento con U-Net**

In [ ]:
# Configuración de rutas

PROJ = Path("/content/drive/MyDrive/geotermia")
IMG_DIR = PROJ / "tiles_stack3"
MSK_DIR = PROJ / "tiles_mask"

# Dataset personalizado

class ChangeDataset(Dataset):
    def __init__(self, files, img_dir, msk_dir):
        self.files = files
        self.img_dir = img_dir
        self.msk_dir = msk_dir
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        name = self.files[i]
        with rasterio.open(self.img_dir/f"{name}.tif") as src:
            x = src.read().astype(np.float32)
        with rasterio.open(self.msk_dir/f"{name}.tif") as src:
            y = src.read(1).astype(np.int64)
        # Normalización simple NDVI [-1,1][0,1]

        x = (x + 1) / 2.0
        return torch.tensor(x), torch.tensor(y)

files = [f.stem for f in IMG_DIR.glob("*.tif")]
print("Total de tiles:", len(files))


#  División entrenamiento/validación

from sklearn.model_selection import train_test_split
train_files, val_files = train_test_split(files, test_size=0.2, random_state=42)

train_set = ChangeDataset(train_files, IMG_DIR, MSK_DIR)
val_set   = ChangeDataset(val_files, IMG_DIR, MSK_DIR)

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=8)


# Modelo U-Net

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Unet(encoder_name="resnet34", in_channels=3, classes=3).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# Entrenamiento

n_epochs = 100
train_losses, val_losses, val_ious = [], [], []

metric = JaccardIndex(task="multiclass", num_classes=3).to(device)

for epoch in range(n_epochs):

    # Entrenamiento
    model.train(); epoch_loss = 0
    for X, y in tqdm(train_loader, desc=f"Época {epoch+1}/{n_epochs}"):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    # Validación

    model.eval(); val_loss = 0; ious = []
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            val_loss += loss_fn(pred, y).item()
            ious.append(metric(pred.argmax(1), y).item())
    val_losses.append(val_loss / len(val_loader))
    val_ious.append(np.mean(ious))

    print(f"Época {epoch+1}: loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}, IoU={val_ious[-1]:.3f}")


#  Guardar modelo

torch.save(model.state_dict(), PROJ/"unet_change_3ch.pt")
print("Modelo guardado en", PROJ/"unet_change_3ch.pt")


# Curvas de pérdida e IoU

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train"); plt.plot(val_losses, label="Val")
plt.title("Pérdida"); plt.legend()
plt.subplot(1,2,2)
plt.plot(val_ious, label="IoU"); plt.legend(); plt.title("IoU (validación)")
plt.show()

# Visualización de predicciones

X, y = val_set[0]
model.eval()
with torch.no_grad():
    pred = model(X.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(y, cmap="gray"); plt.title("Máscara real (0↓,1=,2↑)")
plt.subplot(1,2,2)
plt.imshow(pred, cmap="RdYlGn", vmin=0, vmax=2)
plt.title("Predicción U-Net")
plt.show()
